# 데이터 정렬과 유용한 메서드


**설명 전용 자료:** 개념·예제·주의점을 Markdown 셀로 제공합니다. 연습문제와 빈 풀이 셀은 같은 이름의 `_Practice.ipynb`에 있습니다.

**학습 목표:** 값과 인덱스 정렬을 구분하고 apply·lambda로 값을 변환하며 문자열 분리와 반복 날짜 규칙을 사용합니다.

## 처음 배우는 분을 위한 안내

오름차순은 작은 값부터, 내림차순은 큰 값부터입니다. sort_values는 값으로 정렬하고 sort_index는 행 이름표로 정렬합니다. apply는 각 값에 같은 함수를 적용하고 lambda는 짧은 이름 없는 함수를 뜻합니다. 벡터 연산은 열 전체를 한 번에 계산하는 방법입니다.

**학습 순서:** 문제 1 ~ 6은 필수 기초, 함수·문자열·날짜를 다루는 7 ~ 14는 선택 도전입니다. rrule은 반복 날짜 규칙이며 12 ~ 13번에만 필요합니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

## 예제 데이터

아래 가상 데이터를 공통으로 사용합니다. 코드 블록은 읽기용이며 실행하려면 별도 코드 셀에 복사하세요. 예제는 순서대로 이어집니다. 독립 실행이 필요한 경우 이 준비 코드를 먼저 실행하세요.

```python
import pandas as pd

# 학습용 가상 주문 데이터. 실제 고객 데이터가 아닙니다.
df = pd.DataFrame({
    'order_id': list(range(101, 113)),
    'order_date': ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
                   '2024-01-05', '2024-01-06', '2024-01-07', '2024-02-01',
                   '2024-02-02', '2024-02-03', '2024-02-04', '2024-02-05'],
    'customer_id': [1, 2, 1, 3, 4, 2, 5, 1, 3, 4, 5, 2],
    'product_category': ['Food', 'Book', 'Food', 'Tech', 'Book', 'Tech',
                         'Food', 'Tech', 'Book', 'Food', 'Tech', 'Book'],
    'region': ['South', 'North', 'South', 'East', 'North', 'South',
               'East', 'North', 'South', 'East', 'North', 'South'],
    'quantity': [2, 1, 3, 1, 2, 2, 1, 1, 3, 2, 1, 2],
    'unit_price': [100, 200, 100, 1000, 200, 1000, 100, 1000, 200, 100, 1000, 200],
    'discount': [0, 0.1, 0, 0.2, 0, 0.1, 0, 0, 0.1, 0, 0.2, 0],
    'payment_method': ['Card', 'Wallet', 'Card', 'COD', 'Card', 'Wallet',
                       'COD', 'Card', 'Wallet', 'Card', 'COD', 'Card'],
    'delivery_days': [1, 4, 3, 7, 2, 5, 8, 2, 6, 3, 9, 4],
    'customer_rating': [4.5, 4.0, 4.8, 2.5, 4.2, 3.5, 2.0, 4.6, 3.0, 4.4, 2.2, 4.1],
    'revenue': [200, 180, 300, 800, 400, 1800, 100, 1000, 540, 200, 800, 400]
})
df['order_date'] = pd.to_datetime(df['order_date'])
df_original = df.copy(deep=True)

customer_master = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 6],
    'membership_tier': ['Gold', 'Silver', 'Gold', 'Bronze', 'Bronze'],
    'region': ['South', 'North', 'East', 'North', 'West']
})
customer_summary = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5],
    'total_orders': [3, 3, 2, 2, 2],
    'total_spent': [1500, 2380, 1340, 600, 900],
    'avg_order_value': [500, 2380 / 3, 670, 300, 450]
})

codes = pd.DataFrame({'customer_id': ['CUST-001', 'CUST-002', 'CUST-003']})
date_text = pd.Series(['01/15/2024', '02/03/2024', '12/31/2024'], name='date_text')
```

## sort_values: 값으로 순서 정하기

기본 ascending=True는 오름차순이며 False는 내림차순입니다. 정렬은 순서를 바꾸는 작업이므로 값이나 행을 자동 삭제하지 않습니다. 결측의 위치는 na_position으로 지정할 수 있습니다. 동점의 순서를 확정하려면 order_id 같은 두 번째 기준을 명시하세요.

```python
top = df.sort_values(['revenue', 'order_id'], ascending=[False, True]).head(3)
print(top[['order_id', 'revenue']])  # 106:1800, 108:1000, 104:800
slow = df.sort_values(['delivery_days', 'order_id'], ascending=[False, True])
print(slow.head(3))
```

## 인덱스 보존과 reset_index

sort_values는 기본적으로 원래 행의 인덱스 레이블을 보존합니다. iloc[0]은 정렬 후 첫 위치, loc[0]은 레이블 0인 행입니다. reset_index(drop=True)는 레이블을 새로 붙이지만, 원래 인덱스가 추적에 필요하다면 버리지 않습니다. reset_index()는 원래 인덱스를 열로 보존합니다.

```python
ranked = df.sort_values('revenue', ascending=False)
print(ranked.iloc[0]['order_id'])  # 106
print(ranked.loc[0, 'order_id'])   # 101
ranked_clean = ranked.reset_index(drop=True)
```

## sort_index: 레이블 순서

sort_index는 매출 값이 아니라 행 인덱스의 순서로 정렬합니다. groupby 결과를 지역명순으로 표시하거나 시간 인덱스를 정리할 때 사용합니다. DataFrame 열 이름을 정렬하려면 axis=1을 지정합니다. sort_index는 새 순번 부여 작업과 다릅니다.

```python
totals = df.groupby('region')['revenue'].sum()
print(totals.sort_index())
print(totals.sort_values(ascending=False))
```

## apply와 lambda

lambda x: 식은 하나의 값을 받아 표현식 결과를 반환하는 익명 함수입니다. Series.apply는 각 요소에 함수를 적용합니다. DataFrame.apply의 axis=1은 행별 적용으로 의미가 다릅니다. 간단한 산술은 벡터 연산이 더 읽기 쉽고 보통 효율적입니다. lambda가 항상 더 빠르다는 뜻은 아닙니다. 복잡하고 재사용하는 로직은 def로 이름을 붙이세요.

```python
work = df.copy()
work['revenue_adjusted'] = work['revenue'].apply(lambda x: x * 1.1)
work['discount_percent'] = work['discount'] * 100
work['delivery_type'] = work['delivery_days'].apply(
    lambda days: 'Fast' if days <= 3 else 'Standard')
print(work[['revenue', 'revenue_adjusted', 'delivery_type']].head())
```

## 조건 변환과 결측

조건식의 경계는 문장으로 먼저 정합니다. 3일 이하는 빠른 배송, 3일 초과는 일반 배송입니다. 결측값을 곧바로 일반 배송으로 분류하면 '정보 없음'을 정상으로 잘못 처리할 수 있습니다. 별도로 Unknown을 반환하도록 처리합니다.

```python
def classify_delivery(days):
    if pd.isna(days):
        return 'Unknown'
    return 'Fast' if days <= 3 else 'Standard'

print(pd.Series([3, 4, None]).apply(classify_delivery))
```

## split과 Series.str.split

문자열 하나에는 'CUST-001'.split('-'), 문자열 Series에는 .str.split('-')을 사용합니다. expand=True는 나눈 조각을 열로 펼칩니다. CUST-001의 001은 식별자이므로 숫자로 바꾸면 앞자리 0이 사라집니다. 구분자가 없거나 여러 번 나타나는 값도 고려해야 합니다.

```python
parts = codes['customer_id'].str.split('-', n=1, expand=True)
parts.columns = ['customer_type', 'customer_number']
print(parts)  # CUST / 001 등
date_parts = date_text.str.split('/', expand=True)
date_parts.columns = ['month', 'day', 'year']
print(date_parts)
```

## 날짜를 실제 날짜로 다루기

문자열 분리는 서식이 보장된 경우에만 사용합니다. 날짜 계산·월별 집계에는 to_datetime과 .dt 접근자가 더 적합합니다. 01/02가 1월 2일인지 2월 1일인지 모호하면 format을 명시합니다. 실제 CSV·Excel은 서로 날짜 표현이 다를 수 있으므로 '/', '-'를 무조건 가정하지 마세요.

```python
dates = pd.to_datetime(date_text, format='%m/%d/%Y')
print(dates.dt.year)
print(dates.dt.month)
```

## rrule로 반복 날짜 생성

rrule은 pandas 메서드가 아니라 python-dateutil의 기능입니다. WEEKLY는 주 단위, MO·FR은 요일, dtstart는 시작 경계, count는 생성 개수입니다. 시작일이 조건 요일과 맞으면 포함됩니다. 무한 규칙을 list로 만들지 말고 count나 until로 범위를 제한하세요. 실제 알림이나 메시지를 발송하는 작업은 아닙니다.

```python
from datetime import datetime
from dateutil.rrule import rrule, WEEKLY, MO, FR

dates = list(rrule(WEEKLY, byweekday=MO, dtstart=datetime(2024, 1, 1), count=5))
print([d.strftime('%Y-%m-%d') for d in dates])
# 2024-01-01, 01-08, 01-15, 01-22, 01-29
fridays = list(rrule(WEEKLY, byweekday=FR, dtstart=datetime(2024, 3, 1), count=4))
print([d.strftime('%Y-%m-%d') for d in fridays])
```

## date_range와 rrule 비교

단순 일·주 간격 목록은 pd.date_range가 편합니다. 복수 요일이나 복잡한 달력 규칙은 rrule로 표현할 수 있습니다. date_range도 요일 등 다양한 빈도를 지원하므로 단순히 '규칙은 전부 rrule만 가능'한 것은 아닙니다. 날짜 표시의 요일 이름은 실행 환경의 언어 설정에 따라 달라질 수 있습니다.

```python
mondays = pd.date_range('2024-01-01', periods=5, freq='W-MON')
print(mondays.strftime('%Y-%m-%d').tolist())
```

## 복습 순서

1. 개념과 입력·출력의 분석 단위를 확인합니다.
2. 작은 예제에서 결과를 손으로 예상합니다.
3. 연습용 노트북의 준비 셀을 실행하고 문제를 풉니다.
4. 결과 수치와 점검 기준을 비교하고 이유를 기록합니다.
5. 기본 문제를 마친 뒤 선택 도전을 진행합니다. 입력을 바꾸면 결과도 달라지는 이유를 설명합니다.

## 선택 도전: 동점일 때 정렬 기준 정하기

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
import pandas as pd
extra_df = pd.DataFrame({'order_id':[1,2,3], 'revenue':[100,200,200]})
```

### 진행 순서

1. revenue는 큰 값부터, 같은 매출이면 order_id는 작은 값부터 정렬합니다.
2. order_id 목록을 출력합니다.
3. 두 정렬 기준에 대응하는 ascending 값 두 개를 말로 설명합니다.

**완료 기준:** order_id 순서 [2,3,1]. 매출 내림차순은 False, ID 오름차순은 True.

### 선택 도전 풀이

```python
extra_sorted=extra_df.sort_values(['revenue','order_id'],ascending=[False,True])
print(extra_sorted['order_id'].tolist())
assert extra_sorted['order_id'].tolist()==[2,3,1]
```